# Triage-IN — Phase 1d only: test-set candidates (no GPU needed)

Restores a previous run's `phase1_outputs.zip` (add it via **Add Input → Your Work**), keeps the
test batches already finished by `gemini-3.5-flash`, and fills the rest with `gemini-3.1-flash-lite`.
Training data (1a/1b) is **not** touched.

Settings: Accelerator **None** is fine, Internet **On**, secret `GEMINI_API_KEY` attached.
Run with **Save Version → Save & Run All (Commit)**.

In [ ]:
import glob, os, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("triage-in"):
    subprocess.run(["git", "clone", "-q", "https://github.com/pcd24/triage-in.git"], check=True)
os.chdir("/kaggle/working/triage-in")
!git pull -q && git log --oneline -1
!pip install -q -e . --no-deps && pip install -q regex google-genai==2.33.0
backups = glob.glob("/kaggle/input/**/phase1_outputs.zip", recursive=True)
print("Restoring", backups[0] if backups else "NOTHING - add the previous version's output as input!")
if backups:
    !unzip -o -q {backups[0]} -d /kaggle/working/triage-in
!ls data/raw/test_synthetic/

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")

## 1d. Fill the missing test batches (~25 calls, ~5 min)

In [ ]:
!python -m triage_in.data.generate_test --yes 2>&1 | grep -v -E "httpx|AFC is enabled"
# second pass retries any batch whose reply failed to parse
!python -m triage_in.data.generate_test --yes 2>&1 | grep -v -E "httpx|AFC is enabled" | tail -3

In [ ]:
!cd /kaggle/working/triage-in && zip -q -r -FS /kaggle/working/phase1_outputs.zip data/raw data/interim data/manual -x 'data/*/smoke/*'
!ls -lh /kaggle/working/phase1_outputs.zip
import csv, collections
rows = list(csv.DictReader(open("data/manual/test_synthetic_review.csv", encoding="utf-8-sig")))
print(len(rows), "test candidates;", collections.Counter(r["gen_model"] for r in rows))
print(collections.Counter(r["lang_mix"] for r in rows))